# Predicting Student Health Risk — Local Ensemble Baseline

A local-ready baseline derived from the supplied stacked tree notebook. It uses stratified validation, balanced accuracy, missing-value indicators, lightweight health features, and a probability blend of CatBoost, XGBoost, and LightGBM.

Start with `QUICK_RUN = True`; switch it off for the full training set and final submission.

In [ ]:
from pathlib import Path
import time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
QUICK_RUN = True
QUICK_ROWS = 150_000
DATA_DIR = Path('.')

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET, ID_COL = 'health_condition', 'id'
LABELS = ['unhealthy', 'at-risk', 'fit']
LABEL_TO_INT = {label: i for i, label in enumerate(LABELS)}
INT_TO_LABEL = {i: label for label, i in LABEL_TO_INT.items()}

if QUICK_RUN and len(train) > QUICK_ROWS:
    train, _ = train_test_split(train, train_size=QUICK_ROWS, stratify=train[TARGET], random_state=RANDOM_STATE)
    train = train.reset_index(drop=True)

print(f'Training rows: {len(train):,} | Test rows: {len(test):,}')
display(train[TARGET].value_counts(normalize=True).rename('share').to_frame())

## Feature engineering

In [ ]:
def add_features(df):
    df = df.copy()
    eps = 1e-6
    base = [c for c in df.columns if c not in [ID_COL, TARGET]]
    df['missing_count'] = df[base].isna().sum(axis=1).astype('int8')
    df['bmi_distance_normal'] = (df['bmi'] - 22.0).abs()
    df['sleep_distance_8h'] = (df['sleep_duration'] - 8.0).abs()
    df['steps_per_exercise_min'] = df['step_count'] / (df['exercise_duration'] + eps)
    df['calories_per_step'] = df['calorie_expenditure'] / (df['step_count'] + eps)
    df['water_per_calorie'] = df['water_intake'] / (df['calorie_expenditure'] + eps)
    return df.replace([np.inf, -np.inf], np.nan)

train_fe = add_features(train)
test_fe = add_features(test)
FEATURES = [c for c in test_fe.columns if c != ID_COL]
CAT_COLS = test_fe[FEATURES].select_dtypes(exclude=np.number).columns.tolist()
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
print(f'{len(NUM_COLS)} numerical + {len(CAT_COLS)} categorical features')

## Stratified holdout

In [ ]:
train_idx, valid_idx = train_test_split(
    np.arange(len(train_fe)), test_size=0.20,
    stratify=train_fe[TARGET], random_state=RANDOM_STATE
)
X_train = train_fe.loc[train_idx, FEATURES].copy()
X_valid = train_fe.loc[valid_idx, FEATURES].copy()
y_train = train_fe.loc[train_idx, TARGET].map(LABEL_TO_INT)
y_valid = train_fe.loc[valid_idx, TARGET].map(LABEL_TO_INT)

## CatBoost

In [ ]:
def catboost_frame(df):
    out = df.copy()
    for col in CAT_COLS:
        out[col] = out[col].fillna('<MISSING>').astype(str)
    return out

X_train_cat = catboost_frame(X_train)
X_valid_cat = catboost_frame(X_valid)
test_cat = catboost_frame(test_fe[FEATURES])

cat_model = CatBoostClassifier(
    iterations=1200 if not QUICK_RUN else 500,
    learning_rate=0.05, depth=8, loss_function='MultiClass',
    eval_metric='MultiClass', auto_class_weights='Balanced',
    random_seed=RANDOM_STATE, verbose=100, allow_writing_files=False,
)
cat_model.fit(X_train_cat, y_train, cat_features=CAT_COLS,
              eval_set=(X_valid_cat, y_valid), early_stopping_rounds=100)
cat_valid = cat_model.predict_proba(X_valid_cat)
cat_test = cat_model.predict_proba(test_cat)
print('CatBoost balanced accuracy:', balanced_accuracy_score(y_valid, cat_valid.argmax(1)))

## Shared encoding for XGBoost and LightGBM

In [ ]:
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median', add_indicator=True), NUM_COLS),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                                   encoded_missing_value=-1, dtype=np.float32)),
    ]), CAT_COLS),
], verbose_feature_names_out=False)

X_train_enc = preprocessor.fit_transform(X_train)
X_valid_enc = preprocessor.transform(X_valid)
test_enc = preprocessor.transform(test_fe[FEATURES])
sample_weight = y_train.map((1 / y_train.value_counts(normalize=True)).to_dict()).to_numpy()
print(X_train_enc.shape)

## XGBoost

In [ ]:
xgb_model = XGBClassifier(
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    n_estimators=1000 if not QUICK_RUN else 400, learning_rate=0.05,
    max_depth=8, min_child_weight=15, subsample=.85, colsample_bytree=.8,
    reg_lambda=5, tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_model.fit(X_train_enc, y_train, sample_weight=sample_weight,
              eval_set=[(X_valid_enc, y_valid)], verbose=100)
xgb_valid = xgb_model.predict_proba(X_valid_enc)
xgb_test = xgb_model.predict_proba(test_enc)
print('XGBoost balanced accuracy:', balanced_accuracy_score(y_valid, xgb_valid.argmax(1)))

## LightGBM

In [ ]:
lgb_model = LGBMClassifier(
    objective='multiclass', num_class=3,
    n_estimators=1200 if not QUICK_RUN else 500, learning_rate=0.04,
    num_leaves=63, max_depth=-1, min_child_samples=50,
    subsample=.85, colsample_bytree=.8, reg_lambda=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)
lgb_model.fit(X_train_enc, y_train)
lgb_valid = lgb_model.predict_proba(X_valid_enc)
lgb_test = lgb_model.predict_proba(test_enc)
print('LightGBM balanced accuracy:', balanced_accuracy_score(y_valid, lgb_valid.argmax(1)))

## Select blend weights on validation data

In [ ]:
valid_probs = [cat_valid, xgb_valid, lgb_valid]
test_probs = [cat_test, xgb_test, lgb_test]
model_names = ['CatBoost', 'XGBoost', 'LightGBM']

candidates = []
grid = np.arange(0, 1.01, 0.1)
for w_cat in grid:
    for w_xgb in grid:
        w_lgb = 1 - w_cat - w_xgb
        if w_lgb < -1e-9: continue
        weights = np.array([w_cat, w_xgb, max(0, w_lgb)])
        blend = sum(w*p for w, p in zip(weights, valid_probs))
        score = balanced_accuracy_score(y_valid, blend.argmax(1))
        candidates.append((score, *weights))

best_score, *best_weights = max(candidates)
best_weights = np.array(best_weights)
print('Best balanced accuracy:', round(best_score, 6))
print(dict(zip(model_names, best_weights.round(2))))

valid_blend = sum(w*p for w, p in zip(best_weights, valid_probs))
valid_pred = valid_blend.argmax(1)
print(classification_report(y_valid, valid_pred, target_names=LABELS, digits=4))
display(pd.DataFrame(confusion_matrix(y_valid, valid_pred), index=LABELS, columns=LABELS))

## Create submission

In [ ]:
test_blend = sum(w*p for w, p in zip(best_weights, test_probs))
submission = sample_submission.copy()
submission[TARGET] = pd.Series(test_blend.argmax(1)).map(INT_TO_LABEL)
assert submission.shape == sample_submission.shape
assert submission[TARGET].isin(LABELS).all()
assert submission[ID_COL].equals(test[ID_COL])

submission.to_csv('submission_local_ensemble.csv', index=False)
display(submission[TARGET].value_counts(normalize=True).rename('share').to_frame())
display(submission.head())
print('Saved submission_local_ensemble.csv')

## Next improvements

- Set `QUICK_RUN = False` and rerun on all rows.
- Replace the holdout with 3–5 fold out-of-fold predictions for more reliable blending.
- Tune decision thresholds or class-specific probability multipliers directly for balanced accuracy.
- Validate feature engineering with ablation tests; do not assume every ratio helps.
- Compare against each single model and keep the ensemble only if the gain is repeatable across folds.